## What Is Entity Memory?

### The core idea in one sentence
Instead of remembering only raw conversation text, extract **named entities** and store durable **structured facts** about them.

---

### The mental model — contact cards for the user's world

Imagine FinCoach keeps a small profile card for every important thing the user mentions:

- Chiru
- salary
- monthly expenses
- parents
- emergency fund
- FD
- car goal
- risk profile

Each card accumulates facts over time. When the user asks a new question, FinCoach looks up the relevant cards and injects those facts into the prompt.

Entity Memory is how an agent starts building a structured understanding of the user's world.


In the previous notebook, Vector Store Memory helped us retrieve old conversation turns by semantic similarity.

That is powerful, but it still stores **chunks of text**. If the user says:

> My monthly take-home salary is ₹1,20,000 and I send ₹15,000 to my parents.

A vector store remembers the sentence. Entity Memory extracts the structure:

- `Chiru -> has monthly salary -> ₹1,20,000`
- `Chiru -> supports -> parents`
- `parents -> receive monthly support -> ₹15,000`

This makes memory easier to inspect, update, filter, persist, and inject precisely.


### Key Concepts

**Entity**: A named thing the agent should remember. Examples: a person, goal, account, preference, place, organization, project, expense, or investment.

**Entity extraction**: Using an LLM to identify entities and facts from a conversation turn.

**Entity store**: A structured key-value store where each entity has a name, type, facts, timestamps, and update history.

**Fact merging**: Adding new facts without duplicating old ones. Example: if the user mentions salary twice, keep one clean salary fact.

**Entity context injection**: Looking up relevant entities for the current query and adding their facts to the model prompt.

**Cross-session persistence**: Saving entity facts so the agent can remember stable user information in future sessions.

**Entity drift**: The main failure mode. If extraction is wrong or stale facts are not updated, the agent may confidently use incorrect memory.


### Mermaid source flowchart LR

    U[User message] --> C[Chat model response]
    C --> X[Entity extractor]
    U --> X
    X --> F[Structured entity facts]
    F --> S[(Entity store)]

    Q[Next user query] --> L[Entity lookup]
    S --> L
    L --> P[Prompt builder]
    B[Recent message buffer] --> P
    P --> O[OpenAI chat model]
    O --> A[Assistant reply]

### What this shows

- The assistant responds normally.
- The completed turn is passed to an entity extractor.
- Extracted facts are merged into the entity store.
- Future prompts include only relevant entity facts, not the entire history.


In [ ]:
# Install required packages.
# 'openai'   — OpenAI SDK for chat and entity extraction API calls.
# 'tiktoken' — GPT-4o's exact tokeniser for token counting.
!pip install openai tiktoken --quiet


In [ ]:
# --- Standard library ---
import json                              # For parsing structured extraction output and persistence.
import os                                # For reading environment variables.
import re                                # For simple query/entity matching.
import time                              # For pacing live demos.
from collections import deque            # Recent message buffer with automatic eviction.
from datetime import datetime, timezone  # UTC timestamps, Python 3.12+ compatible.
from typing import List, Dict, Optional  # Type hints for clarity and IDE support.
from dataclasses import dataclass, field, asdict  # Clean data models.

# --- Third-party ---
import openai    # OpenAI SDK.
import tiktoken  # Exact tokeniser for GPT-4o.


In [ ]:
# --- API Key Setup ---
# Option A: Colab Secrets (recommended)
try:
    from google.colab import userdata
    # google.colab is only available in Colab — this import fails locally.
    OPENAI_API_KEY = userdata.get('OPENAI_API_KEY')
    # Fetch the key from Colab's encrypted secrets vault.
    print("Key loaded from Colab Secrets.")
except Exception:
    # Option B: Environment variable (for local Jupyter / VS Code).
    OPENAI_API_KEY = os.environ.get("OPENAI_API_KEY", "")
    print("Key loaded from environment variable.")

# Validate the key looks correct before we try to use it.
assert OPENAI_API_KEY and OPENAI_API_KEY.startswith("sk-"), \
    "API key missing or invalid. Add OPENAI_API_KEY to Colab Secrets or environment."

# Create the OpenAI client — the single object used for all API calls.
client = openai.OpenAI(api_key=OPENAI_API_KEY)

# Model and tokeniser constants.
MODEL = "gpt-4o"
# gpt-4o: GPT-4o — strong general-purpose model for FinCoach responses.

EXTRACTION_MODEL = "gpt-4o"
# Use the same model for extraction so the notebook stays simple.
# In production you may use a cheaper model if quality remains acceptable.

TOKENISER = tiktoken.get_encoding("o200k_base")
# o200k_base is the tokeniser family used by GPT-4o.

print(f"Client ready. Chat model: {MODEL} | Extraction model: {EXTRACTION_MODEL}")


---
## Part 1 — The Message and Entity Data Models

`Message` is the same conversation unit used in earlier notebooks.

`EntityRecord` is the new structure. Each entity has:

- name
- type
- facts
- aliases
- first seen timestamp
- last seen timestamp


In [ ]:
@dataclass
class Message:
    """
    A single conversation message — role, content, and timestamp.
    Same data model as the earlier memory techniques.
    """

    role: str
    # 'user' or 'assistant' — who sent this message.

    content: str
    # The text of the message.

    timestamp: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # UTC timestamp — auto-set to now when the Message is created.

    def to_api_format(self) -> Dict[str, str]:
        """
        Strip the timestamp and return only what the OpenAI API accepts.
        """
        return {"role": self.role, "content": self.content}


@dataclass
class EntityRecord:
    """
    Structured memory record for one entity.
    Think of this as a small durable profile card.
    """

    name: str
    # Canonical entity name, for example: Chiru, Emergency fund, Parents.

    entity_type: str
    # person, goal, preference, finance, expense, investment, organization, place, other.

    facts: List[str] = field(default_factory=list)
    # Durable facts known about this entity.

    aliases: List[str] = field(default_factory=list)
    # Alternate names. Example: Chiru, Chirantan, user.

    first_seen: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # When this entity first entered memory.

    last_seen: str = field(
        default_factory=lambda: datetime.now(timezone.utc).isoformat()
    )
    # When this entity was last updated.

print("Message and EntityRecord dataclasses defined.")


---
## Part 2 — The EntityStore Class

The store is a dictionary keyed by a normalized entity name.

It supports four core operations:

1. create or update an entity
2. merge facts without duplicates
3. search for relevant entities
4. save/load to JSON

In production this might live in Postgres, Redis, a document database, or a managed memory service. Here we keep it visible and inspectable.


In [ ]:
class EntityStore:
    """
    Key-value store mapping entity names to structured records.
    """

    def __init__(self):
        self.entities: Dict[str, EntityRecord] = {}
        # key: normalized entity name, value: EntityRecord.

    # ------------------------------------------------------------------
    # NORMALISATION
    # ------------------------------------------------------------------

    def _key(self, name: str) -> str:
        """
        Normalize entity names so 'Chiru' and ' chiru ' map to one record.
        """
        return re.sub(r"\s+", " ", name.strip().lower())

    # ------------------------------------------------------------------
    # WRITE PATH
    # ------------------------------------------------------------------

    def update_entity(
        self,
        name: str,
        entity_type: str,
        facts: List[str],
        aliases: Optional[List[str]] = None,
    ) -> None:
        """
        Create or update an entity with new facts.
        """

        key = self._key(name)
        if not key:
            return

        aliases = aliases or []

        if key not in self.entities:
            self.entities[key] = EntityRecord(
                name=name.strip(),
                entity_type=entity_type.strip() or "other",
            )
            print(f"  [ENTITY] Created: {name.strip()} ({entity_type})")

        record = self.entities[key]
        record.last_seen = datetime.now(timezone.utc).isoformat()

        if entity_type and record.entity_type == "other":
            record.entity_type = entity_type
            # Upgrade generic type if we later learn a better one.

        for alias in aliases:
            alias = alias.strip()
            if alias and alias not in record.aliases and alias != record.name:
                record.aliases.append(alias)

        for fact in facts:
            clean_fact = fact.strip()
            if clean_fact and clean_fact not in record.facts:
                record.facts.append(clean_fact)
                print(f"    + fact: {clean_fact}")

    # ------------------------------------------------------------------
    # READ PATH
    # ------------------------------------------------------------------

    def search(self, query: str, max_entities: int = 6) -> List[EntityRecord]:
        """
        Find entities relevant to a user query using simple keyword overlap.
        This is intentionally transparent for teaching.
        """

        query_terms = set(re.findall(r"[a-zA-Z0-9₹,]+", query.lower()))
        scored = []

        for record in self.entities.values():
            searchable_text = " ".join(
                [record.name, record.entity_type] + record.aliases + record.facts
            ).lower()
            score = sum(1 for term in query_terms if term in searchable_text)

            # Also boost if the entity name itself appears as a phrase.
            if record.name.lower() in query.lower():
                score += 3

            if score > 0:
                scored.append((score, record))

        scored.sort(key=lambda item: item[0], reverse=True)
        return [record for _, record in scored[:max_entities]]

    def summary(self) -> str:
        """
        Human-readable dump of all known entities.
        """

        if not self.entities:
            return "No entities stored yet."

        blocks = []
        for record in self.entities.values():
            facts = "\n".join(f"  - {fact}" for fact in record.facts) or "  - No facts yet"
            aliases = f" aliases: {', '.join(record.aliases)}" if record.aliases else ""
            blocks.append(f"{record.name} ({record.entity_type}){aliases}\n{facts}")

        return "\n\n".join(blocks)

    # ------------------------------------------------------------------
    # PERSISTENCE
    # ------------------------------------------------------------------

    def save(self, path: str) -> None:
        """
        Persist entity store to a JSON file.
        """

        payload = {key: asdict(record) for key, record in self.entities.items()}
        with open(path, "w", encoding="utf-8") as f:
            json.dump(payload, f, indent=2, ensure_ascii=False)

        print(f"Saved entity store to {path}")

    @classmethod
    def load(cls, path: str) -> "EntityStore":
        """
        Load entity store from a JSON file.
        """

        store = cls()
        with open(path, "r", encoding="utf-8") as f:
            payload = json.load(f)

        for key, record_dict in payload.items():
            store.entities[key] = EntityRecord(**record_dict)

        print(f"Loaded {len(store.entities)} entities from {path}")
        return store

print("EntityStore class defined.")


---
## Part 3 — The EntityExtractor Class

The extractor asks OpenAI to return structured JSON.

For every completed turn, it receives:

- the user message
- the assistant reply

It returns a list of entities, where each entity has:

- `name`
- `type`
- `facts`
- `aliases`

This is the equivalent of function/tool-style extraction, but kept simple and notebook-friendly.


In [ ]:
ENTITY_EXTRACTION_PROMPT = """You extract durable entity memory from a conversation turn.

Return ONLY valid JSON with this shape:
{
  "entities": [
    {
      "name": "canonical entity name",
      "type": "person|goal|preference|finance|expense|investment|organization|place|account|other",
      "facts": ["short durable fact"],
      "aliases": ["optional alias"]
    }
  ]
}

Rules:
- Extract only facts that are useful for future personalization or task continuity.
- Prefer stable facts over one-off phrasing.
- Include financial amounts exactly as written.
- Do not invent facts.
- If there are no useful entities, return {"entities": []}.
"""


class EntityExtractor:
    """
    Uses OpenAI to extract structured entity facts from each completed turn.
    """

    def __init__(self, model: str = EXTRACTION_MODEL):
        self.model = model

    def extract(self, user_message: str, assistant_reply: str) -> List[Dict]:
        """
        Extract entities from one user-assistant exchange.
        """

        extraction_input = {
            "user_message": user_message,
            "assistant_reply": assistant_reply,
        }

        response = client.chat.completions.create(
            model=self.model,
            temperature=0,
            response_format={"type": "json_object"},
            messages=[
                {"role": "system", "content": ENTITY_EXTRACTION_PROMPT},
                {"role": "user", "content": json.dumps(extraction_input, ensure_ascii=False)},
            ],
        )

        raw_json = response.choices[0].message.content
        # JSON-mode asks the model to return valid JSON, but we still parse defensively.

        try:
            parsed = json.loads(raw_json)
        except json.JSONDecodeError:
            print("  [WARN] Entity extractor returned invalid JSON.")
            return []

        entities = parsed.get("entities", [])
        if not isinstance(entities, list):
            return []

        return entities

print("EntityExtractor class defined.")


---
## Part 4 — The EntityMemory Class

`EntityMemory` orchestrates the full loop:

1. find relevant entity facts for the current query
2. build the prompt with entity context and recent messages
3. call GPT-4o
4. extract entities from the completed turn
5. merge extracted facts into the entity store

This gives the agent structured long-term memory without sending the full transcript every time.


In [ ]:
class EntityMemory:
    """
    Chat agent with entity memory: extract, store, retrieve, and inject entity facts.
    """

    # ------------------------------------------------------------------
    # INITIALISATION
    # ------------------------------------------------------------------

    def __init__(
        self,
        session_id: str,
        user_id: str,
        system_prompt: str,
        recent_buffer_turns: int = 3,
        store: Optional[EntityStore] = None,
        extractor: Optional[EntityExtractor] = None,
    ):
        self.session_id = session_id
        # Identifies this conversation session.

        self.user_id = user_id
        # User scope — critical for production memory isolation.

        self.system_prompt = system_prompt
        # FinCoach's persona and rules.

        self.recent_buffer_turns = recent_buffer_turns
        # Number of recent turns kept verbatim.

        self._recent_messages: deque = deque(maxlen=recent_buffer_turns * 2)
        # Short-term conversational flow.

        self.store = store or EntityStore()
        # Structured long-term entity memory.

        self.extractor = extractor or EntityExtractor()
        # OpenAI-powered extraction layer.

        self.turn_count = 0
        # Completed user-assistant turns.

        self.extraction_log: List[Dict] = []
        # Debugging: what was extracted after each turn.

        print(f"[EntityMemory] Initialised — session: {self.session_id}, "
              f"user: {self.user_id}, recent buffer: {self.recent_buffer_turns} turns")

    # ------------------------------------------------------------------
    # ENTITY CONTEXT
    # ------------------------------------------------------------------

    def _format_entity_context(self, query: str) -> str:
        """
        Look up entities relevant to the current query and format them for the prompt.
        """

        relevant_entities = self.store.search(query)

        if not relevant_entities:
            return "No relevant entity memory found."

        blocks = []
        for entity in relevant_entities:
            facts = "\n".join(f"- {fact}" for fact in entity.facts)
            blocks.append(f"{entity.name} ({entity.entity_type})\n{facts}")

        return "\n\n".join(blocks)

    def get_messages_for_api(self, current_user_message: str) -> List[Dict[str, str]]:
        """
        Build the API message list:
        1. system prompt
        2. relevant entity facts
        3. recent verbatim messages
        4. current user message
        """

        entity_context = self._format_entity_context(current_user_message)

        messages = [
            {"role": "system", "content": self.system_prompt},
            {
                "role": "system",
                "content": (
                    "Relevant structured entity memory. Use these facts when helpful. "
                    "If a fact is not listed, do not pretend to remember it.\n\n"
                    f"{entity_context}"
                ),
            },
        ]

        messages.extend(message.to_api_format() for message in self._recent_messages)
        messages.append({"role": "user", "content": current_user_message})

        return messages

    # ------------------------------------------------------------------
    # EXTRACTION AND MERGE
    # ------------------------------------------------------------------

    def _extract_and_store(self, user_message: str, assistant_reply: str) -> None:
        """
        Extract entities from a completed exchange and merge them into the store.
        """

        extracted_entities = self.extractor.extract(user_message, assistant_reply)

        print(f"  [EXTRACT] {len(extracted_entities)} entities found")

        for entity in extracted_entities:
            self.store.update_entity(
                name=entity.get("name", ""),
                entity_type=entity.get("type", "other"),
                facts=entity.get("facts", []),
                aliases=entity.get("aliases", []),
            )

        self.extraction_log.append({
            "turn": self.turn_count,
            "user_message": user_message,
            "entities": extracted_entities,
        })

    # ------------------------------------------------------------------
    # CHAT LOOP
    # ------------------------------------------------------------------

    def chat(self, user_message: str, verbose: bool = True) -> str:
        """
        Send one user message to FinCoach, then extract and store entity facts.
        """

        messages = self.get_messages_for_api(user_message)

        response = client.chat.completions.create(
            model=MODEL,
            max_tokens=1024,
            temperature=0.7,
            messages=messages,
        )

        assistant_reply = response.choices[0].message.content

        self.turn_count += 1
        self._recent_messages.append(Message(role="user", content=user_message))
        self._recent_messages.append(Message(role="assistant", content=assistant_reply))

        self._extract_and_store(user_message, assistant_reply)

        if verbose:
            print(f"[Turn {self.turn_count}] API — prompt: {response.usage.prompt_tokens} tokens, "
                  f"completion: {response.usage.completion_tokens} tokens | "
                  f"known entities: {len(self.store.entities)}")

        return assistant_reply

    # ------------------------------------------------------------------
    # INSPECTION AND PERSISTENCE
    # ------------------------------------------------------------------

    def known_entities(self) -> str:
        """
        Return formatted entity store contents.
        """
        return self.store.summary()

    def save_memory(self, path: str) -> None:
        """
        Persist entity memory to disk.
        """
        self.store.save(path)

    @classmethod
    def from_saved(cls, path: str, **kwargs) -> "EntityMemory":
        """
        Create an EntityMemory instance from a saved entity store.
        """
        store = EntityStore.load(path)
        return cls(store=store, **kwargs)

print("EntityMemory class defined.")


---
## Part 5 — The FinCoach Agent with Entity Memory

The persona stays consistent with the earlier notebooks. The only change is the memory layer.

Entity Memory is especially useful for FinCoach because financial advice depends on stable user-specific facts:

- income
- expenses
- dependents
- goals
- risk profile
- existing investments
- preferences


In [ ]:
# FinCoach's system prompt — same persona as the earlier notebooks.
# Consistent across all techniques — the agent's memory changes, not its identity.
FINCOACH_SYSTEM_PROMPT = """You are FinCoach, a personal financial advisor assistant.
You serve users in India who want guidance on savings, investments, budgeting, and financial planning.

Your principles:
- Always personalise advice using information the user has shared in this conversation.
- Be specific with numbers when the user has provided their financial details.
- Flag when you are making assumptions due to missing information.
- Keep responses concise — 3 to 5 sentences unless the user asks for detail.
- Never provide specific buy/sell recommendations on individual stocks.
- Always recommend consulting a SEBI-registered advisor for major financial decisions.

Use structured entity memory when it is relevant to the user's current question."""

print("FinCoach system prompt defined.")


---
## Part 6 — Demo: Entity Extraction in Action

We run a conversation where the user shares several durable financial facts.

**Watch for:**

- `[EXTRACT]` lines after each turn
- `[ENTITY] Created` lines when new entity cards appear
- `+ fact` lines as facts accumulate
- the final entity store showing a structured profile


In [ ]:
entity_memory = EntityMemory(
    session_id="session_entity_demo",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    recent_buffer_turns=3,
)

demo_turns = [
    "Hi! I'm Chiru. My monthly take-home salary is ₹1,20,000.",
    "My monthly expenses are about ₹60,000 for rent, food, and transport.",
    "I send ₹15,000 to my parents every month.",
    "I have an FD of ₹50,000 that matures in 3 months. I am risk-averse.",
    "My first goal is to build an emergency fund before investing aggressively.",
    "I may buy a car next year, but I do not want a huge EMI.",
    "Given everything you know about me, what should my monthly savings plan look like?",
]

print("ENTITY MEMORY DEMO — FinCoach")
print("Strategy: extract durable facts into structured entity records")
print("Expected: later advice uses salary, expenses, parents, FD, risk profile, and goals")
print("=" * 75)

for i, user_message in enumerate(demo_turns, start=1):
    print(f"\n--- Turn {i} ---")
    print(f"User: {user_message}")
    response = entity_memory.chat(user_message=user_message, verbose=True)
    print(f"FinCoach: {response}")
    time.sleep(1)

print("\n" + "=" * 75)
print("Demo complete.")


---
## Part 7 — Inspecting the Entity Store

Entity memory should be easy to inspect. This is one of its biggest advantages over raw transcript memory.

You can answer questions like:

- What do we know about Chiru?
- What goals has the user mentioned?
- Which facts are stable enough to reuse next session?
- Did extraction duplicate or distort any facts?


In [ ]:
print("=== Entity Store Contents ===\n")
print(entity_memory.known_entities())


In [ ]:
print("=== Extraction Log ===\n")

for entry in entity_memory.extraction_log:
    print(f"Turn {entry['turn']}: {entry['user_message']}")
    if not entry["entities"]:
        print("  No entities extracted.")
    else:
        for entity in entry["entities"]:
            print(f"  - {entity.get('name')} ({entity.get('type')})")
            for fact in entity.get("facts", []):
                print(f"      fact: {fact}")
    print("-" * 75)


---
## Part 8 — Cross-Conversation Persistence

Entity memory becomes much more useful when it survives beyond one chat session.

In production, persistence requires:

- user consent
- encryption
- deletion controls
- data retention policy
- tenant isolation

For this notebook, we save the entity store to JSON and reload it into a new `EntityMemory` object.


In [ ]:
ENTITY_MEMORY_PATH = "fincoach_entity_memory_chiru.json"

entity_memory.save_memory(ENTITY_MEMORY_PATH)

resumed_memory = EntityMemory.from_saved(
    ENTITY_MEMORY_PATH,
    session_id="session_entity_resumed",
    user_id="chiru_001",
    system_prompt=FINCOACH_SYSTEM_PROMPT,
    recent_buffer_turns=3,
)

print("\n=== Resumed Entity Store ===\n")
print(resumed_memory.known_entities())


---
## Part 9 — Entity Memory vs Sliding Window

No API calls — pure visibility analysis.

The question: if a fact is shared early, will the agent still have it later?

Sliding Window Memory keeps facts only if they are recent.
Entity Memory keeps facts if extraction stored them.


In [ ]:
def compare_entity_vs_window(
    fact_turn: int,
    query_turn: int,
    window_size: int,
    fact_extracted: bool = True,
) -> None:
    """
    Compare whether a fact is visible under sliding window vs entity memory.
    """

    turns_since_fact = query_turn - fact_turn
    visible_in_window = turns_since_fact < window_size
    visible_in_entity_memory = fact_extracted

    window_status = "VISIBLE" if visible_in_window else "EVICTED"
    entity_status = "VISIBLE" if visible_in_entity_memory else "MISSING"

    print(f"Fact turn {fact_turn:>2} -> query turn {query_turn:>2} | "
          f"window={window_size:>2} turns | "
          f"Sliding Window: {window_status:<7} | Entity Memory: {entity_status}")


print("=" * 80)
print("FinCoach — Fact Visibility Analysis")
print("=" * 80)

compare_entity_vs_window(fact_turn=1, query_turn=7, window_size=3)  # salary
compare_entity_vs_window(fact_turn=2, query_turn=7, window_size=3)  # expenses
compare_entity_vs_window(fact_turn=3, query_turn=7, window_size=3)  # parents
compare_entity_vs_window(fact_turn=4, query_turn=7, window_size=3)  # FD/risk
compare_entity_vs_window(fact_turn=5, query_turn=7, window_size=3)  # emergency fund

print("\nKey insight:")
print("  Sliding windows forget by age.")
print("  Entity memory forgets only if extraction missed the fact or retention policy deletes it.")


---
## Part 10 — Cost Model: Entity Memory vs Full Buffer

No API calls — pure token arithmetic.

Entity Memory has an extra extraction call after each turn. But the prompt can stay small because you inject a compact set of entity facts instead of the full transcript.

This tradeoff is usually worth it when:

- sessions are long
- facts are reused across sessions
- personalization matters
- raw transcript injection becomes expensive or noisy


In [ ]:
def compare_buffer_vs_entity_cost(
    turns: List[str],
    avg_response_tokens: int = 90,
    avg_entity_context_tokens: int = 180,
    avg_extraction_input_tokens: int = 220,
    avg_extraction_output_tokens: int = 80,
) -> None:
    """
    Compare rough token usage for full buffer vs entity memory.
    This is a planning model, not exact billing math.
    """

    system_tokens = len(TOKENISER.encode(FINCOACH_SYSTEM_PROMPT))
    buffer_history_tokens = 0

    print(f"System prompt: {system_tokens} tokens")
    print()
    print(f"{'Turn':>4} | {'Buffer prompt':>13} | {'Entity prompt':>13} | {'Extraction':>10}")
    print("-" * 62)

    for i, user_msg in enumerate(turns, start=1):
        user_tokens = len(TOKENISER.encode(user_msg))

        buffer_history_tokens += user_tokens
        buffer_prompt = system_tokens + buffer_history_tokens
        buffer_history_tokens += avg_response_tokens

        entity_prompt = system_tokens + avg_entity_context_tokens + user_tokens
        extraction_tokens = avg_extraction_input_tokens + avg_extraction_output_tokens

        print(f"{i:>4} | {buffer_prompt:>13,} | {entity_prompt:>13,} | {extraction_tokens:>10,}")

    print("-" * 62)
    print("Key insight:")
    print("  Entity memory adds extraction work, but prompt size stays bounded and focused.")


simulated_turns = [
    "Hi, I'm Chiru. My monthly salary is ₹1,20,000.",
    "My monthly expenses are ₹60,000.",
    "I send ₹15,000 to my parents every month.",
    "I have an FD of ₹50,000 maturing in 3 months.",
    "I'm risk-averse and prefer stable returns.",
    "My goal is to build an emergency fund first.",
    "I may buy a car next year.",
    "Can you build a monthly plan for me?",
    "How much should I keep liquid?",
    "What should I avoid given my risk profile?",
]

compare_buffer_vs_entity_cost(simulated_turns)


---
## Part 11 — Failure Modes: What Can Go Wrong

Entity Memory is structured, but structure can make mistakes feel more authoritative.

Common failure modes:

1. **Missed extraction** — the user stated a fact but the extractor ignored it.
2. **Bad merge** — the store created duplicates like `Chiru`, `Chirantan`, and `the user`.
3. **Stale fact** — salary or expenses changed but the old value stayed active.
4. **Over-extraction** — the store remembers throwaway details that should not persist.
5. **Privacy risk** — sensitive personal facts are retained without the right controls.

The demo below gives you a simple checklist for reviewing entity quality.


In [ ]:
def audit_entity_store(store: EntityStore) -> None:
    """
    Lightweight quality audit for an entity store.
    """

    print("=== Entity Memory Audit ===")
    print(f"Total entities: {len(store.entities)}")

    possible_duplicates = []
    names = list(store.entities.keys())
    for i, left in enumerate(names):
        for right in names[i + 1:]:
            if left in right or right in left:
                possible_duplicates.append((left, right))

    print(f"Possible duplicate name pairs: {len(possible_duplicates)}")
    for left, right in possible_duplicates[:5]:
        print(f"  - {left} <-> {right}")

    fact_count = sum(len(record.facts) for record in store.entities.values())
    print(f"Total stored facts: {fact_count}")

    entities_without_facts = [record.name for record in store.entities.values() if not record.facts]
    print(f"Entities without facts: {entities_without_facts or 'none'}")


audit_entity_store(entity_memory.store)


---
## Key Takeaways

**What you built:** An `EntityMemory` system with an `EntityStore`, OpenAI-powered `EntityExtractor`, recent message buffer, prompt injection, persistence, and audit helpers.

---

### The progression so far

| | Technique 06 — Vector Store | Technique 07 — Entity Memory |
|---|---|---|
| Stores | Text chunks + embeddings | Structured entity facts |
| Retrieval | Semantic similarity | Entity/fact lookup |
| Best for | Long-range recall | User profile and durable facts |
| Inspectability | Medium | High |
| Main risk | Retrieval miss | Extraction or stale-fact error |
| Production role | Semantic long-term memory | Structured personalization memory |

---

### The three things to remember

1. **Entity Memory turns conversation into structured state.** This is much easier to inspect than a long transcript or vector database.

2. **Extraction quality matters.** Bad extraction creates bad memory. Always log, inspect, and evaluate what the extractor stores.

3. **Persistence changes the privacy bar.** Once memory survives across sessions, you need consent, deletion, retention, and user-level isolation.

---

### What breaks next — and what Technique 08 fixes

Entity Memory stores facts about individual entities, but it does not naturally model relationships between them.

If we want to answer questions like:

- Chiru supports parents every month.
- Parents affect monthly obligations.
- Monthly obligations affect emergency fund target.
- Emergency fund target affects investment capacity.

then we need relationship memory. Technique 08 (Knowledge Graph Memory) stores facts as triples and enables multi-hop reasoning.


### FAQ

Q: What is Entity Memory in agent memory?

A: Entity Memory extracts named entities and durable facts from conversation turns, stores them in a structured entity store, and injects relevant facts into future prompts. It gives the agent a user profile and a structured understanding of important people, goals, preferences, and constraints.

Q: How is Entity Memory different from Vector Store Memory?

A: Vector Store Memory stores text chunks and retrieves them by semantic similarity. Entity Memory stores structured facts about named entities. Vector memory is better for broad semantic recall. Entity memory is better for stable personalization facts like salary, goals, preferences, account names, and important people.

Q: When should I use Entity Memory?

A: Use Entity Memory when the agent needs durable user-specific facts across a session or across multiple sessions. It is especially useful for assistants, coaches, CRM tools, finance planners, healthcare intake systems, project copilots, and any application where personalization depends on stable facts.

Q: What are the main failure modes?

A: The main failure modes are missed extraction, duplicate entities, stale facts, over-retention, and privacy risk. Entity Memory can make wrong facts look official because they are stored in structured form. That is why extraction logs, review tools, and deletion workflows matter.

Q: Should every entity fact persist forever?

A: No. Production systems should classify memory by sensitivity and durability. Some facts should persist, some should expire, and some should never be stored. Users should be able to inspect, edit, and delete long-term memory.

Q: Can Entity Memory and Vector Store Memory work together?

A: Yes. A strong production pattern is to use a recent buffer for immediate context, vector memory for semantic recall, entity memory for durable profile facts, and knowledge graph memory for relationships. Each layer answers a different memory question.
